# SQL 기초 + SQLAlchemy 2.0 ORM

- sqlite3 인메모리 DB로 SQL 4대 명령을 익히고, SQLAlchemy 2.0 ORM으로 Python 클래스와 테이블을 연결합니다.

## 1. SQL 4대 명령

| 명령 | 역할 | 예시 |
|------|------|------|
| `CREATE TABLE` | 테이블 생성 | `CREATE TABLE songs (...)` |
| `INSERT INTO` | 행 삽입 | `INSERT INTO songs VALUES (...)` |
| `SELECT` | 행 조회 | `SELECT * FROM songs` |
| `UPDATE` | 행 수정 | `UPDATE songs SET rank=1 WHERE id=3` |
| `DELETE` | 행 삭제 | `DELETE FROM songs WHERE id=3` |

SQLite는 파일 하나가 곧 데이터베이스입니다. `:memory:` 를 경로로 쓰면 메모리 안에만 존재하는 임시 DB가 됩니다.

In [ ]:
import sqlite3

In [ ]:
# connect(): DB 파일에 연결합니다 (없으면 생성, :memory: 는 인메모리)


In [ ]:
cursor.execute("""
CREATE TABLE songs (
    id      INTEGER PRIMARY KEY AUTOINCREMENT,
    title   TEXT    NOT NULL,
    artist  TEXT    NOT NULL,
    rank    INTEGER
)
""")

In [ ]:
# 멜론 차트 TOP5 삽입
songs_data = [
    ("LOVE ATTACK", "리센느", 1),
    ("갑자기", "아이오아이", 2),
    ("LEMONADE", "aespa", 3),
    ("It's Me", "아일릿", 4),
    ("소문의 낙원", "악뮤", 5),
]

In [ ]:
# executemany(): 여러 행을 한 번에 삽입합니다


In [ ]:
# SELECT 전체 조회

for row in :
    print(f"  [{row[3]}위] {row[2]} - {row[1]}")

In [ ]:
# UPDATE + DELETE


In [ ]:
# SELECT 전체 조회


In [ ]:
# DELETE: 5위 삭제

print(f"\n삭제 후 총 {}곡")

In [ ]:
# SELECT 전체 조회


## 2. ORM 개념 + SQLAlchemy 2.0

- ORM은 Python 클래스와 DB 테이블을 1:1로 연결합니다.

```
Python 클래스 Song   <->  DB 테이블 songs
song.title          <->  songs.title 컬럼
song 인스턴스        <->  songs 의 한 행(row)
```



| 구분 | 이전 스타일 | SQLAlchemy 2.x 권장 스타일 |
|---|---|---|
| Base | `Base = declarative_base()` | `class Base(DeclarativeBase): pass` |
| 컬럼 | `id = Column(Integer, primary_key=True)` | `id: Mapped[int] = mapped_column(primary_key=True)` |
| 전체 조회 | `db.query(User).all()` | `db.scalars(select(User)).all()` |
| 조건 조회 | `db.query(User).filter(...).first()` | `db.scalar(select(User).where(...))` |
| PK 조회 | `db.query(User).filter(User.id == id).first()` | `db.get(User, id)` |



In [ ]:
import sqlalchemy

print("SQLAlchemy 버전:", sqlalchemy.__version__)

### 2.1 ORM 모델 정의 — Song

ORM(Object-Relational Mapping)은 Python 클래스와 DB 테이블을 연결합니다. 이제 앞에서 SQL로 만든 `songs` 테이블을 `Song` 클래스로 표현합니다.

```text
Python                      SQLite
Song 클래스          ↔      songs 테이블
Song.title 속성      ↔      songs.title 컬럼
Song(...) 인스턴스   ↔      songs 테이블의 행 한 개
```

- `__tablename__`: 연결할 DB 테이블 이름입니다.
- `Mapped[int]`, `Mapped[str]`: ORM에 매핑되는 속성의 Python 타입입니다.
- `mapped_column()`: 기본키, 문자열 길이, null 허용 여부 같은 컬럼 규칙을 설정합니다.
- `nullable=False`: 반드시 값이 있어야 하는 `NOT NULL` 컬럼입니다.
- `primary_key=True`: 각 행을 구분하는 기본키입니다.


In [ ]:



class Base():
    pass


class Song(Base):


    # Mapped[int]: SQLAlchemy 2.0 의 타입 어노테이션 방식입니다



    def __repr__(self) -> str:
        return (
            f"Song(id={self.id}, title={self.title!r}, "
            f"artist={self.artist!r}, rank={self.rank})"
        )


print("등록된 테이블:", )


### 2.2 Engine과 Session

### Engine

Engine은 애플리케이션과 DB 사이의 연결 통로입니다. `create_engine()`을 호출했다고 즉시 모든 연결이 열리는 것은 아니며, 실제 쿼리가 필요할 때 연결 풀에서 연결을 사용합니다.

### Session

Session은 ORM 객체를 추적하고 하나의 작업 단위로 DB 변경을 관리합니다.

- `add()`: 객체를 세션에 등록합니다. 아직 DB에 확정된 것은 아닙니다.
- `flush()`: SQL을 전송하지만 트랜잭션을 확정하지 않습니다.
- `commit()`: 트랜잭션을 확정하여 변경을 DB에 반영합니다.
- `refresh()`: DB의 최신 값을 객체에 다시 읽습니다. 자동 생성된 id 확인에 유용합니다.
- `rollback()`: 실패한 트랜잭션을 되돌립니다.
- `close()`: 세션이 사용하던 연결 자원을 반환합니다.

아래 실습은 실행할 때마다 깨끗한 상태가 되도록 인메모리 SQLite를 사용합니다.


In [ ]:


#DATABASE_URL = "sqlite:///./melonchart.db"
DATABASE_URL =

engine =


# create_engine(): DB 연결 엔진을 만듭니다


print("테이블 생성 완료:", )


### 2.3 CREATE와 READ

새 데이터를 등록할 때는 ORM 객체를 만들고 `add()`한 뒤 `commit()`합니다.
`commit()` 전의 객체는 세션이 추적하지만 트랜잭션이 아직 확정되지 않은 상태입니다.

조회는 SQLAlchemy 2.x의 `select()`로 SQL 표현식을 만들고 Session을 통해 실행합니다.

- `db.scalars(select(Song)).all()`: Song 객체 여러 개
- `db.scalar(select(Song).where(...))`: Song 객체 한 개 또는 `None`
- `db.get(Song, 1)`: 기본키가 1인 Song 객체 또는 `None`


In [ ]:
'''
Song(title="Supernatural", artist="NewJeans",    rank=1),
Song(title="Kitsch",       artist="IVE",         rank=2),
Song(title="Queencard",    artist="(여자)아이들",  rank=3),
Song(title="Hype Boy",     artist="NewJeans",    rank=4),
Song(title="After LIKE",   artist="IVE",         rank=5),
'''




### 2.4 UPDATE와 DELETE

ORM에서는 UPDATE SQL 문자열을 직접 작성하지 않아도 됩니다. 조회한 객체의 속성을 바꾸면 Session이 변경을 감지하고, `commit()` 시 UPDATE를 실행합니다.

삭제도 동일합니다. 객체를 조회한 뒤 `delete()`에 전달하고 `commit()`합니다.

> 반드시 `None` 여부를 검사하세요. 존재하지 않는 객체의 속성을 변경하거나 `delete(None)`을 실행하면 서버는 의도하지 않은 500 오류를 반환합니다. FastAPI에서는 이 상황을 404로 변환해야 합니다.


In [ ]:
with SessionLocal() as db:
    # UPDATE



    # DELETE — 현재 5위 곡 삭제


    print(f"삭제 후 {len()}곡:", )


## 3. Artist 1 : N Song 관계

한 아티스트가 여러 곡을 발표할 수 있으므로 `Artist 1 : N Song` 관계가 됩니다. 관계형 DB에서는 곡 테이블의 `artist_id` 외래키로 소유 아티스트를 표현합니다.

```text
Artist: IVE
  ├── Kitsch
  ├── After LIKE
  └── I AM
```

- `ForeignKey("artists.id")`: 곡이 참조하는 아티스트 기본키입니다.
- `song.artist`: 곡에서 아티스트 객체로 이동합니다.
- `artist.songs`: 아티스트에서 곡 목록으로 이동합니다.
- `back_populates`: 관계의 양쪽 속성을 연결합니다.
- `selectinload()`: 관계 데이터를 한꺼번에 로드하여 N+1 쿼리를 줄입니다.


In [ ]:
from sqlalchemy import ForeignKey, Integer, String, select
from sqlalchemy.orm import Mapped, mapped_column, relationship, selectinload


# =========================================================
# 1. ORM 모델 정의
# =========================================================
class Artist(Base):
    __tablename__ = "artists"

    id: Mapped[int] = mapped_column(
        primary_key=True,
        autoincrement=True,
    )
    name: Mapped[str] = mapped_column(
        String(100),
        nullable=False,
        unique=True,
    )

    songs: Mapped[list["ArtistSong"]] = relationship(
        back_populates="artist",
        cascade="all, delete-orphan",
    )


class ArtistSong(Base):
    __tablename__ = "artist_songs"

    id: Mapped[int] = mapped_column(
        primary_key=True,
        autoincrement=True,
    )
    title: Mapped[str] = mapped_column(
        String(200),
        nullable=False,
    )
    rank: Mapped[int | None] = mapped_column(
        Integer,
        nullable=True,
    )

    artist_id: Mapped[int] = mapped_column(
        ForeignKey("artists.id"),
        nullable=False,
    )

    artist: Mapped["Artist"] = relationship(
        back_populates="songs",
    )


# =========================================================
# 2. 새 테이블 생성
# =========================================================


print("현재 등록된 테이블:", )

In [ ]:
# =========================================================
# 3. 가수와 노래 데이터 등록
# =========================================================
with SessionLocal() as db:
    # Artist 데이터가 없을 때만 예제 데이터를 등록합니다.
    # 주피터 셀을 다시 실행할 때 발생할 수 있는 중복 등록을 방지합니다.
    artist_exists = db.scalar(
        select(Artist).limit(1)
    )

    if artist_exists is None:
        ive =
        newjeans =



        # flush()는 아직 commit하지 않은 상태에서 INSERT를 실행합니다.
        # 이를 통해 DB가 자동 생성한 ive.id, newjeans.id를 받을 수 있습니다.
        db.flush()

        print("IVE ID:", )
        print("NewJeans ID:", )

        db.add_all([
            ArtistSong(title="Kitsch", rank=2, artist_id=ive.id),
            ArtistSong(title="After LIKE", rank=5, artist_id=ive.id),
            ArtistSong(title="I AM", rank=7, artist_id=ive.id),
            ArtistSong(title="Supernatural", rank=1, artist_id=newjeans.id),
            ArtistSong(title="Hype Boy", rank=4, artist_id=newjeans.id),
        ])
        db.commit()

        print("예제 데이터 등록 완료")
    else:
        print("이미 등록된 가수 데이터가 있어 등록을 생략합니다.")

In [ ]:
# =========================================================
# 4. 가수와 노래 관계 조회
# =========================================================
with SessionLocal() as db:
    # selectinload()를 사용하여 가수와 연결된 노래를 함께 조회합니다.
    # 가수별로 노래를 반복 조회하는 N+1 문제를 줄일 수 있습니다.
    stmt = (

    )

    # SQLAlchemy 2.x 권장 조회 방식
    artists =

    for artist in artists:
        print(f"\n{artist.name} ({len(artist.songs)}곡)")

        # 순위가 없는 노래는 가장 뒤에 배치합니다.
        sorted_songs =

        for song in sorted_songs:
            rank_text = (
                f"{song.rank}위"
                if song.rank is not None
                else "순위 없음"
            )

            print(f"  [{rank_text}] {song.title}")

---

## 4. CRUD 전체 흐름 정리

| 작업 | SQL 의미 | SQLAlchemy 2.x 코드 | commit 필요 |
|---|---|---|---|
| 생성 | INSERT | `db.add(obj)` | 예 |
| 전체 조회 | SELECT | `db.scalars(select(Model)).all()` | 아니요 |
| 조건 단건 조회 | SELECT + WHERE | `db.scalar(select(Model).where(...))` | 아니요 |
| PK 조회 | SELECT + PK | `db.get(Model, id)` | 아니요 |
| 수정 | UPDATE | 객체 속성 변경 | 예 |
| 삭제 | DELETE | `db.delete(obj)` | 예 |

### 자주 발생하는 실수

- `add()`만 하고 `commit()`하지 않아 서버 재시작 후 데이터가 사라집니다.
- 조회 결과가 `None`인데 속성을 변경하여 500 오류가 발생합니다.
- 전역 Session 하나를 여러 요청이 공유합니다.
- ORM 객체를 응답하면서 Pydantic에 `from_attributes=True`를 설정하지 않습니다.
- 모델에 컬럼을 추가하면 `create_all()`이 기존 테이블도 자동 변경한다고 오해합니다.
- SQLite 상대 경로가 Python 파일 기준이라고 오해합니다. `./`는 실행 작업 폴더 기준입니다.


## 5. 실습 문제

1. `Song`에 `genre: Mapped[str | None]` 컬럼을 추가하세요.
2. 아티스트 이름으로 곡을 검색하는 조회문을 작성하세요. 힌트: `Song.artist.contains(keyword)`
3. 곡이 없을 때 404를 반환하는 `DELETE /songs/{song_id}`를 작성하세요.
4. `PATCH /songs/{song_id}`로 순위만 변경하는 API를 작성하세요.
5. 같은 제목과 아티스트의 곡이 이미 있으면 409를 반환하세요.
6. `Artist`를 삭제했을 때 연결된 `ArtistSong`도 삭제되는지 확인하세요.
7. `engine`에 `echo=True`를 적용하고 각 CRUD에서 실행되는 SQL을 기록하세요.
